In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, fbeta_score, roc_auc_score
)

# Column groups — same as 03_modeling
numeric_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses', 'weight_recorded'
]
binary_cols = ['gender', 'change', 'diabetesMed']
already_binary = ['metformin', 'glipizide', 'glyburide', 'pioglitazone',
                   'rosiglitazone', 'glimepiride', 'repaglinide', 'nateglinide']
ordinal_cols = ['age']
age_order = [['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
              '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']]
nominal_cols = [
    'race', 'max_glu_serum', 'A1Cresult',
    'medical_specialty', 'payer_code',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'diag_1', 'diag_2', 'diag_3',
    'insulin'
]

In [2]:
# load pre transformed data
df_train = pd.read_csv('../data/df_train_v2.csv')
df_test = pd.read_csv('../data/df_test_v2.csv')

print(f"df_train: {df_train.shape}")
print(f"df_test:  {df_test.shape}")

# Sanity check the transformations persisted
print(f"\ndiag_1 unique values: {df_train['diag_1'].nunique()} (expect 9)")
print(f"metformin dtype: {df_train['metformin'].dtype} (expect int)")

df_train: (78283, 36)
df_test:  (19539, 36)

diag_1 unique values: 9 (expect 9)
metformin dtype: int64 (expect int)


In [3]:
# Buils the pipeline and encode
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('bin', OrdinalEncoder(), binary_cols),
        ('ord', OrdinalEncoder(categories=age_order), ordinal_cols),
        ('nom', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), nominal_cols),
        ('passthrough_binary', 'passthrough', already_binary),
    ],
    remainder='drop',
    verbose_feature_names_out=True,
)

# Separate features and target
X_train = df_train.drop(columns=['encounter_id', 'patient_nbr', 'target'])
y_train = df_train['target']
X_test = df_test.drop(columns=['encounter_id', 'patient_nbr', 'target'])
y_test = df_test['target']

# Fit and transform
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print(f"X_train encoded: {X_train_encoded.shape}")
print(f"Number of features: {len(feature_names)}")

X_train encoded: (78283, 189)
Number of features: 189


/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [3, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [4]:
baseline_lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    solver='lbfgs',
    penalty='l2',
    C=1.0
)
baseline_lr.fit(X_train_encoded, y_train)

# Quick performance check
y_test_proba = baseline_lr.predict_proba(X_test_encoded)[:, 1]
auc = roc_auc_score(y_test, y_test_proba)

print(f"L2 Baseline AUC on test: {auc:.4f}")
print(f"(For reference, the 217-feature baseline had AUC ~0.664)")

/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


L2 Baseline AUC on test: 0.6635
(For reference, the 217-feature baseline had AUC ~0.664)


In [5]:
# Extract coefficients from the fitted model
coefficients = baseline_lr.coef_[0]  # shape (n_features,)

# Build a DataFrame for inspection
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False).reset_index(drop=True)

# Show the top 20 features by absolute coefficient magnitude
print("Top 20 features by |coefficient|:")
print(coef_df.head(20).to_string(index=False))

Top 20 features by |coefficient|:
                                        feature  coefficient  abs_coefficient
                nom__discharge_disposition_id_9     1.841017         1.841017
              nom__medical_specialty_Gynecology    -1.744054         1.744054
               nom__discharge_disposition_id_28     1.689007         1.689007
               nom__discharge_disposition_id_15     1.603634         1.603634
nom__medical_specialty_Pediatrics-Endocrinology    -1.524221         1.524221
               nom__discharge_disposition_id_22     1.389310         1.389310
              nom__medical_specialty_Hematology     1.330903         1.330903
          nom__medical_specialty_Otolaryngology    -1.138761         1.138761
                       nom__admission_type_id_7    -1.114536         1.114536
                nom__discharge_disposition_id_5     1.076965         1.076965
      nom__medical_specialty_InfectiousDiseases     1.006675         1.006675
               nom__discharge_

I checked feature importance, but the rankings looked clinically wrong, which led me to discover that rare one-hot dummies were getting large coefficients fit to small patient counts.

In [6]:
# Look at the prevalence of every one-hot encoded column
# (Numeric features and binary passthroughs will show their own values, not prevalence)

X_train_df = pd.DataFrame(X_train_encoded, columns=feature_names)

# Prevalence = fraction of rows where the feature equals 1
# This is meaningful for one-hot dummies; numeric features will give different values
prevalence = (X_train_df != 0).mean()

# Focus on nom__ (one-hot) features
nom_features = [f for f in feature_names if f.startswith('nom__')]
nom_prevalence = prevalence[nom_features].sort_values()

print(f"Total one-hot features: {len(nom_features)}")
print(f"\nLowest-prevalence one-hot dummies (smallest patient counts):")
print(nom_prevalence.head(20))

Total one-hot features: 168

Lowest-prevalence one-hot dummies (smallest patient counts):
nom__medical_specialty_Neurophysiology                     0.000013
nom__medical_specialty_Perinatology                        0.000013
nom__medical_specialty_Proctology                          0.000013
nom__medical_specialty_Psychiatry-Addictive                0.000013
nom__medical_specialty_Pediatrics-InfectiousDiseases       0.000013
nom__medical_specialty_Surgery-PlasticwithinHeadandNeck    0.000013
nom__medical_specialty_SportsMedicine                      0.000013
nom__medical_specialty_Speech                              0.000013
nom__payer_code_FR                                         0.000013
nom__admission_source_id_13                                0.000013
nom__medical_specialty_Pediatrics-Hematology-Oncology      0.000026
nom__medical_specialty_Pediatrics-AllergyandImmunology     0.000026
nom__medical_specialty_Resident                            0.000026
nom__admission_source_id_2

In [7]:
import matplotlib.pyplot as plt

# How many features fall at each prevalence level?
prevalence_bins = [0, 0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
prevalence_labels = ['<0.1%', '0.1-0.5%', '0.5-1%', '1-5%', '5-10%', 
                     '10-25%', '25-50%', '50%+']

# Bucket each one-hot feature by prevalence
buckets = pd.cut(nom_prevalence, bins=prevalence_bins, labels=prevalence_labels, include_lowest=True)
print("Distribution of one-hot features by prevalence bucket:")
print(buckets.value_counts().sort_index())

# Cumulative count below each threshold
print(f"\nCumulative count of features below each threshold:")
for t in [0.001, 0.005, 0.01, 0.02, 0.05]:
    count = (nom_prevalence < t).sum()
    print(f"  < {t*100:.1f}% prevalence: {count} features ({count/len(nom_prevalence)*100:.1f}% of one-hot dummies)")

Distribution of one-hot features by prevalence bucket:
<0.1%       63
0.1-0.5%    20
0.5-1%      14
1-5%        34
5-10%       16
10-25%      10
25-50%       7
50%+         4
Name: count, dtype: int64

Cumulative count of features below each threshold:
  < 0.1% prevalence: 63 features (37.5% of one-hot dummies)
  < 0.5% prevalence: 83 features (49.4% of one-hot dummies)
  < 1.0% prevalence: 97 features (57.7% of one-hot dummies)
  < 2.0% prevalence: 111 features (66.1% of one-hot dummies)
  < 5.0% prevalence: 131 features (78.0% of one-hot dummies)


In [8]:
def prune_and_evaluate(X_train_enc, X_test_enc, y_train, y_test, feature_names, threshold):
    """Prune one-hot features below prevalence threshold, train LR, return metrics."""
    
    # Compute prevalence for each feature
    X_train_df = pd.DataFrame(X_train_enc, columns=feature_names)
    prevalence = (X_train_df != 0).mean()
    
    # Keep features where:
    # - Not a one-hot dummy (numeric, binary, ordinal — keep all), OR
    # - One-hot dummy with prevalence >= threshold
    keep_mask = []
    for name in feature_names:
        if name.startswith('nom__'):
            # One-hot dummy — apply threshold
            keep_mask.append(prevalence[name] >= threshold)
        else:
            # Numeric, binary, ordinal, passthrough — always keep
            keep_mask.append(True)
    
    keep_mask = np.array(keep_mask)
    n_kept = keep_mask.sum()
    n_dropped = (~keep_mask).sum()
    
    # Apply pruning
    X_train_pruned = X_train_enc[:, keep_mask]
    X_test_pruned = X_test_enc[:, keep_mask]
    
    # Train fresh LR
    model = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
        solver='lbfgs',
        penalty='l2',
        C=1.0
    )
    model.fit(X_train_pruned, y_train)
    
    # Evaluate
    y_proba = model.predict_proba(X_test_pruned)[:, 1]
    y_pred = model.predict(X_test_pruned)
    
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f2 = fbeta_score(y_test, y_pred, beta=2)
    
    return {
        'threshold': threshold,
        'n_features': n_kept,
        'n_dropped': n_dropped,
        'auc': auc,
        'precision': prec,
        'recall': rec,
        'f2': f2,
        'kept_features': np.array(feature_names)[keep_mask]
    }


# Baseline (no pruning) for reference
baseline_result = prune_and_evaluate(
    X_train_encoded, X_test_encoded, y_train, y_test, feature_names, threshold=0.0
)

# Moderate prune: drop below 0.5%
moderate_result = prune_and_evaluate(
    X_train_encoded, X_test_encoded, y_train, y_test, feature_names, threshold=0.005
)

# Aggressive prune: drop below 1%
aggressive_result = prune_and_evaluate(
    X_train_encoded, X_test_encoded, y_train, y_test, feature_names, threshold=0.01
)

# Print comparison
print(f"{'Model':<25} {'Features':<10} {'Dropped':<10} {'AUC':<8} {'Precision':<10} {'Recall':<8} {'F2':<8}")
print("=" * 85)
for r in [baseline_result, moderate_result, aggressive_result]:
    label = f"threshold = {r['threshold']*100:.1f}%"
    print(f"{label:<25} {r['n_features']:<10} {r['n_dropped']:<10} {r['auc']:<8.4f} {r['precision']:<10.4f} {r['recall']:<8.4f} {r['f2']:<8.4f}")

/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in versi

Model                     Features   Dropped    AUC      Precision  Recall   F2      
threshold = 0.0%          189        0          0.6635   0.1896     0.5310   0.3904  
threshold = 0.5%          106        83         0.6606   0.1909     0.5387   0.3948  
threshold = 1.0%          92         97         0.6606   0.1914     0.5383   0.3951  


In [9]:
# Save the pruning decision artifacts for next session
pruned_feature_names = aggressive_result['kept_features']
np.save('../data/pruned_feature_names.npy', pruned_feature_names)

# Re-create the pruned encoded matrices and save
X_train_df = pd.DataFrame(X_train_encoded, columns=feature_names)
prevalence = (X_train_df != 0).mean()
keep_mask = np.array([
    (not name.startswith('nom__')) or (prevalence[name] >= 0.01)
    for name in feature_names
])
np.save('../data/X_train_pruned.npy', X_train_encoded[:, keep_mask])
np.save('../data/X_test_pruned.npy', X_test_encoded[:, keep_mask])

print(f"Saved pruned matrices: {keep_mask.sum()} features kept, {(~keep_mask).sum()} dropped")

Saved pruned matrices: 92 features kept, 97 dropped
